# LC 207 — Course Schedule
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Graphs
**Pattern:** Topological Sort / Cycle Detection (Kahn's)

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Model prerequisites
as a directed graph. If the graph has a cycle,
courses cannot all be finished. Use Kahn's
algorithm: process nodes with in-degree 0 first;
if all nodes are processed, no cycle exists.
</div>

## Official Problem Statement

There are a total of `numCourses` courses to take,
labeled from `0` to `numCourses - 1`. You are
given an array `prerequisites` where
`prerequisites[i] = [ai, bi]` indicates that you
must take course `bi` first if you want to take
course `ai`.

Return `true` if you can finish all courses.
Otherwise, return `false`.

**Example 1:**
```
Input:  numCourses=2, prerequisites=[[1,0]]
Output: true
Explanation: take 0 then 1.
```
**Example 2:**
```
Input:  numCourses=2, prerequisites=[[1,0],[0,1]]
Output: false
Explanation: 0 requires 1 and 1 requires 0 — cycle.
```

**Constraints:**
- `1 <= numCourses <= 2000`
- `0 <= prerequisites.length <= 5000`
- `prerequisites[i].length == 2`
- All pairs are unique

## What This Is Actually Asking

Some courses have prerequisites. Is it possible
to take all courses without getting into a loop
where course A requires B and B requires A?
This reduces to: does the prerequisite graph
contain a cycle?

## Walk Through an Example by Hand

```
numCourses=4
prerequisites=[[1,0],[2,0],[3,1],[3,2]]

Graph (a <- b means b is prerequisite of a):
  0 -> 1 -> 3
  0 -> 2 -> 3

In-degrees:
  0: 0   1: 1   2: 1   3: 2

Kahn's — start with in-degree 0 nodes:
  queue=[0]  processed=0

  Pop 0: processed=1
    neighbours: 1, 2
    indegree[1] = 1-1 = 0 -> queue=[1]
    indegree[2] = 1-1 = 0 -> queue=[1,2]

  Pop 1: processed=2
    neighbours: 3
    indegree[3] = 2-1 = 1  (still blocked)

  Pop 2: processed=3
    neighbours: 3
    indegree[3] = 1-1 = 0 -> queue=[3]

  Pop 3: processed=4

processed == numCourses -> no cycle -> True
```

## The Picture

```
Cycle exists — impossible:    No cycle — possible:

  0 --> 1                       0 --> 1 --> 3
  ^     |                       |           ^
  |_____|                       v           |
  (cycle)                       2 ----------+

Kahn's Algorithm — BFS topological sort:

  1. Build adjacency list + in-degree array
  2. Queue all nodes with in-degree = 0
     (no prerequisites — can take immediately)
  3. Pop from queue:
     for each neighbour: in-degree -= 1
     if neighbour in-degree hits 0: add to queue
  4. Count processed nodes.
     processed == numCourses? -> no cycle -> True
     else -> cycle -> False

Cycle means: some nodes keep in-degree > 0 forever
because they depend on each other circularly.
```

## When To Use This Pattern

- When asked **"can all tasks complete given
  dependencies?"**, think **topological sort /
  cycle detection**
- When building in-degrees, think
  **`[ai, bi]` means bi → ai edge, ai's in-degree +1**
- When `processed < numNodes` at end, think
  **cycle detected — return False**
- When all nodes processed, think **True — valid
  topological order exists**

## The Approach

Build an adjacency list and an in-degree count
array from the prerequisites. Add all courses
with in-degree zero to a queue. Process each
course: for every course it unlocks, decrement
that course's in-degree and enqueue it if zero.
Return True if the total courses processed equals
numCourses.

In [ ]:
from collections import deque  # BFS queue for Kahn's
from typing import List

In [ ]:
def test_harness(func):
    tests = [
        # (numCourses, prerequisites, expected)
        (2, [[1,0]],              True),
        (2, [[1,0],[0,1]],        False),  # cycle
        (1, [],                   True),   # single course
        (4, [[1,0],[2,0],[3,1],[3,2]], True),
        (3, [[0,1],[0,2],[1,2]],  True),   # diamond
        (3, [[0,1],[1,2],[2,0]],  False),  # 3-cycle
        (5, [],                   True),   # no prereqs
        (6, [[1,0],[2,1],[3,2],[4,3],[5,4]], True),  # chain
    ]

    passed = 0
    for i, (n, pre, expected) in enumerate(tests):
        result = func(n, [p[:] for p in pre])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"n={n} pre={pre} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def canFinish(
    numCourses: int,
    prerequisites: List[List[int]]
) -> bool:
    """
    Return True if all courses can be finished.

    Build adjacency list and in-degree array. BFS
    (Kahn's): queue all in-degree-0 nodes; pop each,
    decrement neighbours' in-degrees, enqueue those
    that hit 0. Return processed == numCourses.

    Time:  O(V + E) — vertices + edges
    Space: O(V + E) — adjacency list + in-degree array
    """
    pass


# Quick debug — run this cell while building
print(canFinish(2, [[1,0]]))          # True
print(canFinish(2, [[1,0],[0,1]]))    # False
print(canFinish(4, [[1,0],[2,0],[3,1],[3,2]]))  # True
print(canFinish(3, [[0,1],[1,2],[2,0]]))  # False

In [ ]:
# Uncomment and run when solution is ready
# test_harness(canFinish)

## Complexity

| Approach | Time | Space |
|---|---|---|
| DFS cycle detection (visited states) | O(V+E) | O(V+E) |
| Kahn's BFS topological sort | O(V+E) | O(V+E) |

Both approaches are equivalent in complexity.
Kahn's is iterative (no recursion stack overflow)
and directly tells you the processing order as
a side effect.

## Real World Connection

At Citi, the ETL pipeline has dozens of jobs with
strict dependency ordering — job B cannot start
until job A completes. Before deploying a new
dependency, the scheduler runs topological sort
on the DAG to verify no circular dependency has
been introduced.
Course Schedule is the exact check: if Kahn's
algorithm processes fewer jobs than exist, a cycle
was introduced and the deployment is rejected.
Apache Airflow uses the same cycle detection
internally when a DAG file is parsed.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra